# Imports

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

from uavloc.features.imagenet import quantised_rgb
from uavloc.features.segmentation import SEGMENTERS, build_segmenter
from uavloc.segmentation.eval import HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS
from uavloc.segmentation.losses import (CURRENT_LOSS, TRACK, StructureLossConfig, building_ids, ce_dice_loss,
                                        class_skeleton, gap_band, instance_weights, structure_loss_terms)
from uavloc.segmentation.topology import track_scores
from uavloc.track import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
DATASET = "ua-village-annotated"
SAT_CROP_PX = 512
BATCH = 8
RARE_CROP_P = 0.3
GAP_PX = 64
SEED = 0
FLOAT_TOLERANCE = 1e-5
STRUCTURE_LOSS = StructureLossConfig(gap_weight=3.0, instance_weight=1.0, tversky_weight=1.0, skeleton_weight=1.0,
                                     track_tversky_weight=1.0)
PERFECT_LOGIT = 20.0
CUT_EVERY_PX = 40
FIGURE_CROPS = 4

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sampler = vs.SatelliteSampler([s for s in sat if not s.banned.all()], SAT_CROP_PX, RARE_CROP_P)
x, y = vs.batch_of(sampler, np.random.default_rng(SEED), BATCH)
torch.manual_seed(SEED)
logits = torch.randn(BATCH, 6, SAT_CROP_PX, SAT_CROP_PX)
tuple(x.shape)

# Run

In [ ]:
current = structure_loss_terms(logits, y, CURRENT_LOSS)["loss"].item()
reference = ce_dice_loss(logits, y).item()
assert abs(current - reference) < FLOAT_TOLERANCE
{"current loss": current, "ce_dice_loss": reference, "difference": current - reference}

In [ ]:
ids = building_ids(y)
weights = instance_weights(ids, STRUCTURE_LOSS.instance_ref_px, STRUCTURE_LOSS.instance_max_ratio)
assert weights[ids == 0].max() == 0
{"buildings": int(sum(len(torch.unique(c)) - 1 for c in ids)), "weight min": weights[ids > 0].min().item(),
 "weight median": weights[ids > 0].median().item(), "weight max": weights[ids > 0].max().item()}

In [ ]:
assert torch.equal(gap_band(y, STRUCTURE_LOSS.gap_radius_px), gap_band(y, STRUCTURE_LOSS.gap_radius_px, ids=ids))
{k: round(v.item(), 4) for k, v in structure_loss_terms(logits, y, STRUCTURE_LOSS).items()}

In [ ]:
perfect = torch.nn.functional.one_hot(y.clamp(max=5), 6).permute(0, 3, 1, 2).float() * PERFECT_LOGIT
skeleton_loss = structure_loss_terms(perfect, y, STRUCTURE_LOSS)["skeleton_recall_track"].item()
assert skeleton_loss < FLOAT_TOLERANCE
{"skeleton recall loss of a perfect prediction": skeleton_loss}

In [ ]:
def cut(label: np.ndarray, every_px: int) -> np.ndarray:
    """`label` with one-pixel background columns every `every_px`, which breaks every horizontal track."""
    out = label.copy()
    out[:, ::every_px] = 0
    return out

In [ ]:
labels = list(y.numpy())
{"label vs itself": track_scores(labels, labels),
 "cut every 40 px vs label": track_scores([cut(t, CUT_EVERY_PX) for t in labels], labels)}

In [ ]:
{name: tuple(build_segmenter(name).eval()(x[:1]).shape) for name in SEGMENTERS}

# Plots

In [ ]:
band = gap_band(y, STRUCTURE_LOSS.gap_radius_px, ids=ids)
skeleton = class_skeleton(y, TRACK, STRUCTURE_LOSS.skeleton_dilation_px)
fig, axes = plt.subplots(FIGURE_CROPS, 4, figsize=(15, 3.6 * FIGURE_CROPS))
for row, k in enumerate(np.argsort([-(ids[i] > 0).sum().item() for i in range(BATCH)])[:FIGURE_CROPS]):
    axes[row, 0].imshow(quantised_rgb(x[k].numpy()))
    image = axes[row, 1].imshow(weights[k], cmap="magma", vmin=0, vmax=STRUCTURE_LOSS.instance_max_ratio)
    axes[row, 2].imshow(band[k], cmap="gray")
    axes[row, 3].imshow(np.where(skeleton[k], 2, (y[k] == TRACK).numpy()), cmap="viridis", vmin=0, vmax=2)
    fig.colorbar(image, ax=axes[row, 1], fraction=0.046)
for ax, title in zip(axes[0], ("crop", "per-building weight min(1046 / area, 10)", "gap band (r = 10 px)",
                             "track (mid) and its tubed centreline (bright)")):
    ax.set_title(title)
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()